# 01 — Schema Quality

**Purpose:** Verifies all required columns exist, audits nulls, checks coordinate ranges, validates shot types, and removes incomplete or corrupted rallies.

**Outputs:**
- `EDA/data/shuttleset_clean.parquet` — strokes with bad rallies removed
- `EDA/data/removed_rallies.csv` — log of every removed rally and why

In [2]:
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("__file__").resolve().parent))
from config import OUTPUT_DIR, SHOT_TYPE_MAP

In [3]:
RALLY_KEY = ["match_id", "set_num", "rally"]

EXPECTED_COLS = [
    "rally", "ball_round", "time", "frame_num",
    "roundscore_A", "roundscore_B",
    "player", "server", "type",
    "aroundhead", "backhand",
    "hit_height", "hit_area", "hit_x", "hit_y",
    "landing_height", "landing_area", "landing_x", "landing_y",
    "lose_reason", "getpoint_player",
    "player_location_x", "player_location_y",
    "opponent_location_x", "opponent_location_y",
    "match_id", "set_num",
]

In [4]:
# Load Data

def load():
    path = OUTPUT_DIR / "shuttleset_master.parquet"
    if not path.exists():
        print("ERROR: shuttleset_master.parquet not found.")
        print("Run step00_load_data.py first.")
        sys.exit(1)
    df = pd.read_parquet(path)
    print(f"Loaded: {len(df):,} strokes, {df['match_id'].nunique()} matches")
    return df

In [5]:
# Schema check

def check_schema(df: pd.DataFrame):
    print("**SCHEMA CHECK**")

    present = [c for c in EXPECTED_COLS if c in df.columns]
    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    extra   = [c for c in df.columns if c not in EXPECTED_COLS]

    print(f" Expected columns present: {len(present)} / {len(EXPECTED_COLS)}")
    if missing:
        print(f"Missing columns: {missing}")
    if extra:
        print(f"Extra columns (not in spec): {extra}")

In [6]:
# Null audit

def audit_nulls(df: pd.DataFrame):
    print("**NULL AUDIT**")

    cols_to_check = [c for c in EXPECTED_COLS if c in df.columns]
    null_counts = df[cols_to_check].isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    report = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
    report = report[report["null_count"] > 0].sort_values("null_count", ascending=False)

    if report.empty:
        print("No nulls found.")
    else:
        print(report.to_string())
        print("""
Expected nulls (these are normal):
  lose_reason / getpoint_player  → only on the last stroke of each rally
  aroundhead / backhand          → blank when False, 1.0 when True
  hit_x / hit_y                  → blank for serve strokes
""")


In [7]:
# Shot type validation

def check_shot_types(df: pd.DataFrame):
    print("**SHOT TYPE CHECK**")

    found    = set(df["type"].dropna().unique())
    expected = set(SHOT_TYPE_MAP.keys())
    unknown  = found - expected

    print(f"Distinct shot types found: {len(found)} (expected up to 18)")
    if unknown:
        print(f"Unknown types (not in translation map): {unknown}")
    else:
        print("All shot types match the known 18-class schema")

    print("\nShot type distribution (English):")
    dist = df["type"].map(SHOT_TYPE_MAP).value_counts()
    print(dist.to_string())


In [8]:
# Coordinate range check

def check_coordinates(df: pd.DataFrame):
    print(f"\n{'─'*50}")
    print("COORDINATE RANGES")
    print(f"{'─'*50}")

    coord_cols = [
        c for c in ["hit_x", "hit_y", "landing_x", "landing_y",
                     "player_location_x", "player_location_y",
                     "opponent_location_x", "opponent_location_y"]
        if c in df.columns
    ]
    stats = df[coord_cols].describe().loc[["min", "max", "mean"]].round(1)
    print(stats.to_string())


In [9]:
# Flag and remove bad rallies

def remove_bad_rallies(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("**RALLY QUALITY FILTERING**")

    removal_batches = []

    # Rule 1: No winner recorded on last stroke
    last_strokes = df.groupby(RALLY_KEY).last().reset_index()
    no_winner = last_strokes[last_strokes["getpoint_player"].isna()][RALLY_KEY].copy()
    no_winner["removal_reason"] = "no getpoint_player on last stroke"
    removal_batches.append(no_winner)
    print(f"  Rallies with no winner:               {len(no_winner)}")

    # Rule 2: Fewer than 2 strokes
    lengths = df.groupby(RALLY_KEY).size().reset_index(name="n")
    too_short = lengths[lengths["n"] < 2][RALLY_KEY].copy()
    too_short["removal_reason"] = "fewer than 2 strokes"
    removal_batches.append(too_short)
    print(f"  Rallies with fewer than 2 strokes:    {len(too_short)}")

    # Rule 3: Missing player position coordinates
    if "player_location_x" in df.columns and "player_location_y" in df.columns:
        missing_pos = df[
            df["player_location_x"].isna() | df["player_location_y"].isna()
        ][RALLY_KEY].drop_duplicates().copy()
        missing_pos["removal_reason"] = "missing player position coordinates"
        removal_batches.append(missing_pos)
        print(f"  Rallies missing player coordinates: {len(missing_pos)}")

    # Rule 4: Missing landing coordinates
    if "landing_x" in df.columns and "landing_y" in df.columns:
        missing_land = df[
            df["landing_x"].isna() | df["landing_y"].isna()
        ][RALLY_KEY].drop_duplicates().copy()
        missing_land["removal_reason"] = "missing landing coordinates"
        removal_batches.append(missing_land)
        print(f"  Rallies missing landing coordinates:  {len(missing_land)}")

    # Aggregate all removal reasons per rally
    removed_df = pd.concat(removal_batches, ignore_index=True)
    removed_agg = (
        removed_df.groupby(RALLY_KEY)["removal_reason"]
        .apply(lambda x: " | ".join(sorted(set(x))))
        .reset_index()
    )

    print(f"\n  Total rallies to remove: {len(removed_agg)}")

    bad_keys = set(
        zip(removed_agg["match_id"], removed_agg["set_num"], removed_agg["rally"])
    )
    df_clean = df[
        ~df.apply(
            lambda r: (r["match_id"], r["set_num"], r["rally"]) in bad_keys,
            axis=1
        )
    ].copy()

    return df_clean, removed_agg

In [10]:
# Save Cleaned Data

def save_outputs(df_clean: pd.DataFrame, removed: pd.DataFrame):
    # Add English shot type column
    df_clean["type_en"] = df_clean["type"].map(SHOT_TYPE_MAP)

    clean_path   = OUTPUT_DIR / "shuttleset_clean.parquet"
    removed_path = OUTPUT_DIR / "removed_rallies.csv"

    df_clean.to_parquet(clean_path, index=False)
    removed.to_csv(removed_path, index=False)

    print(f"\nSaved: {clean_path}")
    print(f"Saved: {removed_path}")

In [12]:
# Main

def main():
    print("**SCHEMA VERIFICATION & DATA QUALITY**")

    df = load()
    check_schema(df)
    audit_nulls(df)
    check_shot_types(df)
    check_coordinates(df)

    df_clean, removed = remove_bad_rallies(df)

    original_rallies = df.groupby(RALLY_KEY).ngroups
    clean_rallies    = df_clean.groupby(RALLY_KEY).ngroups

    print("**SUMMARY**")
    print(f"  Before: {len(df):,} strokes across {original_rallies:,} rallies")
    print(f"  After:  {len(df_clean):,} strokes across {clean_rallies:,} rallies")
    print(f"  Removed:{len(df) - len(df_clean):,} strokes ({len(removed)} rallies)")

    save_outputs(df_clean, removed)

In [13]:
if __name__ == "__main__":
    main()

**SCHEMA VERIFICATION & DATA QUALITY**
Loaded: 36,484 strokes, 44 matches
**SCHEMA CHECK**
 Expected columns present: 27 / 27
Extra columns (not in spec): ['win_reason', 'flaw', 'player_location_area', 'opponent_location_area', 'db', 'stroke_id']
**NULL AUDIT**
                     null_count  null_%
aroundhead                33148   90.86
lose_reason               32975   90.38
getpoint_player           32974   90.38
backhand                  22785   62.45
landing_height             6318   17.32
hit_x                      4749   13.02
hit_y                      4749   13.02
hit_area                   4703   12.89
landing_x                  1187    3.25
landing_y                  1187    3.25
landing_area               1186    3.25
opponent_location_x        1173    3.22
opponent_location_y        1173    3.22
player_location_x          1170    3.21
player_location_y          1170    3.21
hit_height                   42    0.12

Expected nulls (these are normal):
  lose_reason / getpoi